# HIXL 典型问题分析定位

本节将结合实际案例介绍 HIXL 应用开发中两类典型报错场景的定位分析方法：建链失败类问题和传输失败类问题，涵盖关键错误信息和日志的获取方式、问题定位流程和分析方法等内容。学完后你将能够自行根据报错信息分析并解决应用开发过程中的问题。

本节学习大纲如下：

- HIXL 问题定位整体流程
- HIXL 关键报错信息获取方式
- HIXL 典型问题定位案例


# 1. 问题定位整体流程

通常情况下，当使用 HIXL 进行应用开发的过程中发生了报错，可以按照如下流程由浅入深地排查问题根因：

<img src="./images/troubleshooting_flowchart.png" alt="问题定位整体流程" width="100%">

**注意**：前面的章节提到，HIXL 应用开发可以分为 client 侧和 server 侧，因此定位问题时应该同时结合两端的日志来定位，不能只关注一侧的报错。

建议首先报错信息初步判断出报错根因在 client 还是在 server 侧，然后再深入定位分析实际报错端的报错原因，可以有效提升问题定位的效率。


## 2. 关键报错信息获取方式

当 HIXL 接口调用失败时，错误信息会通过日志输出。定位问题的第一步是获取相关报错堆栈和日志并理解其结构。

### 2.1 获取错误码和堆栈信息

调用 HIXL 接口报错时，其内部会上报错误信息，包含错误码、接口调用堆栈和相关日志参数打印，可以通过`aclGetRecentErrMsg`接口获取，获取示例如下：

```cpp
auto ret = hixl_engine.Initialize(local_engine, options);
if (ret != SUCCESS) {
    printf("[ERROR] Initialize failed, ret = %u, errmsg: %s\n", ret, aclGetRecentErrMsg());
    return -1;
}
```

### 2.2 获取详细报错日志

当从 HIXL 的错误堆栈无法定位问题时，需要进一步获取底层 HCCL 或 RUNTIME 相关日志，一般情况下，应用类日志默认保存在 `$HOME/ascend/log/plog` ，更多日志介绍请查看 [日志参考](https://www.hiascend.com/document/detail/zh/CANNCommunityEdition/900beta2/maintenref/logreference/logreference_0001.html)

```
$HOME/ascend/log/plog/
```

**日志分析原则**： 优先定位首条致命错误，日志中的首条 `[ERROR]` 通常是报错的根本原因，后续错误多为连锁反应。

### 2.3 HIXL 错误码

错误码类型为`uint32_t`，接口调用成功返回`0`，调用失败返回其他值，具体含义如下：

| 枚举值 | 错误码 | 含义 | 是否可恢复 | 解决办法 |
| --- | --- | --- | --- | --- |
| SUCCESS | 0 | 成功 | 无 |  |
| PARAM_INVALID | 103900 | 参数错误 | 是 | 基于日志排查错误原因。 |
| TIMEOUT | 103901 | 处理超时 | 否 | 保留现场，获取 Host/Device 日志，并备份。 |
| NOT_CONNECTED | 103902 | 没有建链 | 是 | 上层排查建链情况。 |
| ALREADY_CONNECTED | 103903 | 已经建链 | 是 | 上层排查建链情况。 |
| NOTIFY_FAILED | 103904 | 通知失败 | 否 | 预留错误码，暂不会返回。 |
| UNSUPPORTED | 103905 | 不支持的参数或接口 | 是 | 预留错误码，暂不会返回。 |
| FAILED | 503900 | 通用失败 | 否 | 保留现场，获取 Host/Device 日志，并备份。 |
| RESOURCE_EXHAUSTED | 203900 | 资源耗尽，当前仅包含 stream 资源 | 是 | 等待资源释放后再进行尝试。 |


## 3. 典型问题定位案例

本节通过“建链失败”这类典型问题的实际案例，带你深入体验以下 HIXL 问题定位的流程。

> 本节构造的错误案例均通过`01.04`章节中的样例改造而来。

如果你想了解更多常见问题的定位分析方法，可以进一步学习 [《HIXL常见问题定位手册》](https://gitcode.com/cann/hixl/wiki/HIXL%E5%B8%B8%E8%A7%81%E9%97%AE%E9%A2%98%E5%AE%9A%E4%BD%8D%E6%89%8B%E5%86%8C.md)。


### 3.1 建链超时时间非法

#### 3.1.1 错误代码示例

client 侧调用建链接口时，传入错误的建链超时时间`0`，导致建链报错，代码示例如下：

```cpp
hixl_ret = engine.Connect(remote_engine, 0);  # 这里超时时间0为非法值
if (hixl_ret != SUCCESS) {
std::cout << "[CLIENT ERROR] Connect failed, ret = " << hixl_ret << ", err_msg: " << aclGetRecentErrMsg() << std::endl;
ClientFinalize(engine, connected, remote_engine, handle, src);
return -1;
}
```

#### 3.1.2 报错现象

执行样例，打屏日志出现如下报错：
```text
[CLIENT INFO] Initialize success
[CLIENT INFO] RegisterMem success
[CLIENT ERROR] Connect failed, ret = 103900, err_msg: E19999: Inner Error!
E19999[PID: 1340518]:  timeout_in_millis:0 must > 0[FUNC:Connect][FILE:hixl_impl.cc][LINE:235]
```

#### 3.1.3 问题分析

对于一些接口使用方式错误（如传参错误等）类型的问题，在异常分支通过`aclGetRecentErrMsg`打印的错误信息能够很好地帮助我们定位问题。

在上面的案例中，我们在调用`Connect`接口失败时，通过`aclGetRecentErrMsg`打印出了关键错误信息`timeout_in_millis:0 must > 0`，从而可以判断，这里的建链失败是由于传入了错误超时时间`0`导致的。

#### 3.1.4 解决方案

根据报错信息可以，建链传入的超时时间必须大于 0，因此修改超时时间或者使用默认值 1000ms。


### 3.2 建链两端链路不一致

#### 3.2.1 错误代码示例

HIXL 支持通过 `HCCL_INTRA_ROCE_ENABLE` 环境变量配置走 RoCE 或 HCCS 通信链路，当 client 和 server 配置走不同的通信链路时，会导致建链报错，代码示例如下：

```bash
# server 侧走 HCCS 链路
./server &

# client 侧走 RoCE 链路
HCCL_INTRA_ROCE_ENABLE=1 ./client &
```

#### 3.2.2 报错现象

执行样例，打屏日志出现如下报错：
```text
[CLIENT ERROR] Connect failed, ret = 103901, err_msg: E19999: Inner Error!
E19999[PID: 1353502]:  Call llm::HcclAdapter::GetInstance().HcclCommPrepare(channel_info_.comm, &prepareConfig, channel_info_.timeout_sec) fail, ret: 0x9[FUNC:PrepareHcclComm][FILE:channel.cc][LINE:103]
        TraceBack (most recent call last):
       Call PrepareHcclComm(hccl_start) fail. Failed to prepare hccl comm[FUNC:Initialize][FILE:channel.cc][LINE:59]
       Call channel->Initialize(enable_use_fabric_mem) fail. Failed to init channel[FUNC:CreateChannel][FILE:channel_manager.cc][LINE:416]
       Call channel_manager_->CreateChannel(channel_info, channel, enable_use_fabric_mem_) fail. [FUNC:CreateChannel][FILE:channel_msg_handler.cc][LINE:337]
       Call CreateChannel(channel_info, is_client, peer_channel_info) fail. Failed to create channel[FUNC:ConnectInfoProcess][FILE:channel_msg_handler.cc][LINE:411]
       Call status.error_code fail. Failed to check peer process ret status, error code[103901], err msg[], local engine:127.0.0.1, remote engine:127.0.0.1:16789, timeout:1000 ms.[FUNC:DoConnect][FILE:channel_msg_handler.cc][LINE:598]
       Call DoConnect(remote_engine, timeout_in_millis) fail. Failed to connect, local engine:127.0.0.1, remote engine:127.0.0.1:16789, timeout:1000 ms.[FUNC:Connect][FILE:channel_msg_handler.cc][LINE:537]
       Call msg_handler_.Connect(remote_engine.GetString(), timeout_in_millis) fail. Failed to connect, remote engine:127.0.0.1:16789, timeout:1000 ms[FUNC:Connect][FILE:adxl_inner_engine.cc][LINE:431]
       Call engine_->Connect(remote_engine, timeout_in_millis) fail. Failed to connect[FUNC:Connect][FILE:hixl_impl.cc][LINE:117]
       Failed to connect, remote engine:127.0.0.1:16789, timeout:1000 ms[FUNC:Connect][FILE:hixl_impl.cc][LINE:237]
```

通过打屏的报错信息只能看出是调用`HcclCommPrepare`接口失败了，看不出具体的报错原因，需要结合 CANN 日志进一步分析。

通过`grep -nr ERROR $HOME/ascend/log/plog/`查看 CANN 日志目录下的报错，会发现如下结构的典型错误日志：

```bash
[ERROR] HCCL(669593,python3):2025-12-17-18:13:19.031.224 [hccl_socket_manager.cc:816] [673943][Wait][LinkEstablish]wait socket establish timeout, role[0] rank[0] timeout[10 s]
[ERROR] HCCL(669593,python3):2025-12-17-18:13:19.031.580 [hccl_socket_manager.cc:883] [673943][Wait][LinksEstablishCompleted] is failed. ret[9].
[ERROR] HCCL(669593,python3):2025-12-17-18:13:19.031.583 [hccl_socket_manager.cc:510] [673943][Create][Sockets]Wait links establish completed failed, local role is client. ret[9]
[ERROR] HCCL(669593,python3):2025-12-17-18:13:19.031.743 [hccl_socket_manager.cc:642] [673943]   _________________________LINK_ERROR_INFO___________________________
[ERROR] HCCL(669593,python3):2025-12-17-18:13:19.031.746 [hccl_socket_manager.cc:643] [673943]   |  comm error, device[0]
[ERROR] HCCL(669593,python3):2025-12-17-18:13:19.031.748 [hccl_socket_manager.cc:645] [673943]   |  dest_ip(user_rank)  |   dest_port   |  src_ip(user_rank)   |   src_port   |   MyRole   |   Status   |    TlsStatus   |
[ERROR] HCCL(669593,python3):2025-12-17-18:13:19.031.750 [hccl_socket_manager.cc:647] [673943]   |----------------------|---------------|----------------------|--------------|------------|------------|----------------|
[ERROR] HCCL(669593,python3):2025-12-17-18:13:19.031.756 [hccl_socket_manager.cc:599] [673943]   |  192.43.2.197(1)   |  16666  |   192.43.2.199(0)   |  0  |  server  | time out |   DISABLE  |
[ERROR] HCCL(669593,python3):2025-12-17-18:13:19.032.113 [hccl_one_sided_conn.cc:68] [673943][Connect]call trace: hcclRet -> 9
[ERROR] HCCL(669593,python3):2025-12-17-18:13:19.032.116 [hccl_one_sided_conn.cc:366] [673943][ConnectWithRemote]call trace: hcclRet -> 9
[ERROR] HCCL(669593,python3):2025-12-17-18:13:19.032.127 [hccl_one_sided_service.cc:699] [673943][ConnectByThread] Connect failed. userrank[0], ret[9].
[ERROR] HCCL(669593,python3):2025-12-17-18:13:19.032.524 [hccl_one_sided_service.cc:743] [671395][HcclOneSidedService][CreateLinkFullmesh] Create links failed. commIdentifier[141.61.29.108:20311_141.61.29.108:21035].
[ERROR] HCCL(669593,python3):2025-12-17-18:13:19.032.539 [hccl_one_sided_service.cc:608] [671395][PrepareFullMesh]call trace: hcclRet -> 9
[ERROR] HCCL(669593,python3):2025-12-17-18:13:19.032.580 [hccl_one_sided_service.cc:653] [671395][HcclOneSidedService][Prepare] Prepare failed. commIdentifier[141.61.29.108:20311_141.61.29.108:21035]
[ERROR] HCCL(669593,python3):2025-12-17-18:13:19.032.583 [one_sided_service_adapt.cc:586] [671395][HcclCommPrepare]call trace: hcclRet -> 9
```

日志中 `LINK_ERROR_INFO` 区域展示了链路的关键信息：

| 字段 | 说明 |
| --- | --- |
| dest_ip(user_rank) | 目标端 IP 和 rank |
| dest_port | 目标端端口 |
| src_ip(user_rank) | 本端 IP 和 rank |
| src_port | 本端端口 |
| MyRole | 角色（server/client） |
| Status | 当前状态 |
| TlsStatus | TLS 配置状态 |

#### 3.2.3 问题分析

建链报错一般需要同时结合 client 侧和 server 侧报错日志中的`LINK_ERROR_INFO`分析。当两侧使用不同的通信链路时，`LINK_ERROR_INFO`打印出来的 ip 会不一致，例如：

sever 侧走 HCCS 链路，`LINK_ERROR_INFO`信息如下：
```text
_________________________LINK_ERROR_INFO___________________________
|  comm error, device[0]
|  dest_ip(user_rank)  |   dest_port   |  src_ip(user_rank)   |   src_port   |   MyRole   |   Status   |    TlsStatus   |
|----------------------|---------------|----------------------|--------------|------------|------------|----------------|
|  192.43.0.197(1)   |  16666  |   192.43.2.199(0)   |  0  |  server  | time out |   DISABLE  | LinkInfo
```

client 侧走 RoCE 链路，`LINK_ERROR_INFO`信息如下：
```text
_________________________LINK_ERROR_INFO___________________________
|  comm error, device[1]
|  dest_ip(user_rank)  |   dest_port   |  src_ip(user_rank)   |   src_port   |   MyRole   |   Status   |    TlsStatus   |
|----------------------|---------------|----------------------|--------------|------------|------------|----------------|
|  6.100.213.101(0)   |  16666  |   6.100.209.44(1)   |  0  |  client  | time out |   DISABLE  | LinkInfo
```

通过两侧打印的`LINK_ERROR_INFO`信息可以看出，两侧的 ip 地址不一致，这种情况一般就是由于两侧使用的链路不一致导致的。

#### 3.2.4 解决方案

排查建链两端是否配置了 `HCCL_INTRA_ROCE_ENABLE` 环境变量，两端需要同时配置为 1 或同时都不配置。


---

## 课后练习

本节介绍了 HIXL 问题定位的整体流程，并通过两个建链失败案例说明日志分析方法，请根据学习内容完成以下题目进行自测。

1. （判断题）当 HIXL 接口调用失败时，可以通过 `aclGetRecentErrMsg` 获取错误码、接口调用堆栈和相关日志参数打印。

2. （判断题）建链报错时，只要 client 侧已经打印出 TraceBack，就无需结合 server 侧日志继续分析。

3. （单选题）当仅从 HIXL 的错误堆栈仍无法定位问题根因时，下一步最适合进一步查看哪个目录下的底层日志？
    A. `$HOME/ascend/log/plog/`
    B. `/var/log/messages`
    C. `/usr/local/Ascend/driver/log/`
    D. `/tmp/hixl/`

4. （单选题）在“建链超时时间非法”案例中，以下哪条报错信息最直接表明根因是建链超时时间传参错误？
    A. `timeout_in_millis:0 must > 0`
    B. `ret = 103901`
    C. `Wait links establish completed failed`
    D. `TlsStatus DISABLE`

5. （多选题）在“建链两端链路不一致”案例中，以下哪些分析结论或处理方法是正确的？
    A. 可对比两侧 `LINK_ERROR_INFO` 中的 IP 信息判断链路是否一致
    B. 两端 `HCCL_INTRA_ROCE_ENABLE` 需要同时配置为 1 或同时都不配置
    C. 只要 `dest_port` 一样，就能排除链路配置问题
    D. `LINK_ERROR_INFO` 中的 `MyRole`、`Status`、`TlsStatus` 等字段有助于定位问题

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/04.02_answer.txt